# BRCA2 Saturation Scoring Fixture

This notebook shows the mechanics for the BRCA2 saturation tutorial on a small committed fixture. It enumerates every possible SNV across a short BRCA2 exon-scale sequence, computes deterministic fixture scores, and renders a calibrated-surprise heatmap. It does not use a corrected released GenoLeWM scorer or the Sahu et al. assay rows from MaveDB `urn:mavedb:00001242-a-1`, so it does not satisfy the full-cohort benchmark criterion in issue #95.

In [1]:
from __future__ import annotations

import os
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "geno_lewm").exists():
            return candidate
    raise RuntimeError("run this notebook from inside the GenoLeWM checkout")


ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(ROOT)

from examples.brca2_saturation_fixture import (  # noqa: E402
    BRCA2_EXON_FIXTURE,
    BRCA2_FIXTURE_CHROM,
    BRCA2_FIXTURE_START_BP,
    enumerate_fixture_saturation,
    render_html_heatmap,
    render_text_heatmap,
    summarize_rows,
)

print(f"repo root: {ROOT.relative_to(ROOT)}")
print(
    f"fixture length={len(BRCA2_EXON_FIXTURE)} bp "
    f"start={BRCA2_FIXTURE_CHROM}:{BRCA2_FIXTURE_START_BP}"
)

repo root: .
fixture length=24 bp start=chr13:32316461


The helper below enumerates three alternate bases at every position. The score columns are deterministic fixture values shaped like the real tutorial output, not model results.

In [2]:
rows = enumerate_fixture_saturation()
summary = summarize_rows(rows)

print(
    f"fixture rows={summary['snvs']} positions={summary['positions']} "
    f"mean_sigma={summary['mean_sigma_calibrated']} "
    f"fixture_spearman={summary['fixture_spearman']}"
)
print("Sahu assay comparison: not run in this fixture notebook")
print("first variants:")
for row in rows[:5]:
    print(
        f"  {row.variant} sigma={row.sigma_calibrated:.3f} "
        f"fixture_function={row.fixture_function_score:.3f}"
    )

fixture rows=72 positions=24 mean_sigma=0.577 fixture_spearman=-0.987
Sahu assay comparison: not run in this fixture notebook
first variants:
  chr13:32316461:A>C sigma=0.367 fixture_function=0.427
  chr13:32316461:A>G sigma=0.422 fixture_function=0.375
  chr13:32316461:A>T sigma=0.568 fixture_function=0.045
  chr13:32316462:T>A sigma=0.278 fixture_function=0.596
  chr13:32316462:T>C sigma=0.302 fixture_function=0.571


The text heatmap uses `x` for the reference base at each position and darker ASCII shades for higher calibrated surprise. In Jupyter, `html_heatmap` renders the same matrix as a styled table without adding plotting dependencies.

In [3]:
text_heatmap = render_text_heatmap(rows)
html_heatmap = render_html_heatmap(rows)

print(text_heatmap)
print(f"heatmap cells={html_heatmap.count('<td style=')}")

try:
    from IPython.display import HTML, display
except ModuleNotFoundError:
    HTML = str

    def display(_value: str) -> None:
        return None


display(HTML(html_heatmap))

ref A T G G A T T T A T C T G C T C T T C G C G T T
pos 0 1 2 3 4 5 6 7 8 9 0 1 2 3 4 5 6 7 8 9 0 1 2 3
A   x - : : x - = - x = = = - = = + + + + + + + * *
C   - - = = = = = = + = x = + x + x + + x * x # * *
G   = + x x = + + + + * * * x * # # # # # x # x % #
T   + x + * * x x x # x * x # * x # x x # % # % x x
heatmap cells=72
